In [31]:
import numpy as np
import random


GRID_SIZE = 8
OBSTACLES = [(1, 1), (4, 1), (2, 3), (5, 3), (5, 4), (5, 5)]
GOAL_POSITION = (0, 7)

robu_row = 7
robu_col = 0

def reset(episode_num):
    """Resets the agent's position systematically based on the episode number."""
    global robu_row, robu_col
    
    # Deterministic rotation to ensure all 64 states are explored
    state_to_try = episode_num % 64
    r = state_to_try // GRID_SIZE
    c = state_to_try % GRID_SIZE
    
    # Safety Check: If it lands on a Goal or Obstacle, send to default start (7,0)
    if (r, c) == GOAL_POSITION or (r, c) in OBSTACLES:
        robu_row, robu_col = 7, 0
    else:
        robu_row, robu_col = r, c
        
    return robu_row * GRID_SIZE + robu_col

def step(action):
    """Executes the action, handles boundaries, and returns (next_state, reward, done)."""
    global robu_row, robu_col
    
    # Action Logic: 0: UP, 1: DOWN, 2: LEFT, 3: RIGHT
    if action == 0 and robu_row > 0:             robu_row -= 1
    elif action == 1 and robu_row < GRID_SIZE-1: robu_row += 1
    elif action == 2 and robu_col > 0:             robu_col -= 1
    elif action == 3 and robu_col < GRID_SIZE-1: robu_col += 1
    
    next_state = robu_row * GRID_SIZE + robu_col
    done = False
    
    # Reward Design
    if (robu_row, robu_col) == GOAL_POSITION:
        reward = 100
        done = True
    elif (robu_row, robu_col) in OBSTACLES:
        reward = -100
        done = True
    else:
        reward = -1  # Step penalty to encourage efficiency
        
    return next_state, reward, done

In [32]:
# Initialize Q-Table (64 States x 4 Actions)
Q = np.zeros((64, 4))

def choose_action(state, epsilon):
    """Selects an action using the Epsilon-Greedy strategy considering valid boundaries."""
    valid_acts = []
    r, c = state // GRID_SIZE, state % GRID_SIZE
    
    # Filter valid moves based on current grid borders
    if r > 0: valid_acts.append(0)           # UP
    if r < GRID_SIZE-1: valid_acts.append(1)  # DOWN
    if c > 0: valid_acts.append(2)           # LEFT
    if c < GRID_SIZE-1: valid_acts.append(3)  # RIGHT

    # Exploration vs Exploitation Choice
    if random.uniform(0, 1) < epsilon:
        return random.choice(valid_acts)     # Explore: Random move
    else:
        # Exploit: Select the action with the highest Q-value out of valid actions
        q_values = [Q[state, a] for a in valid_acts]
        max_idx = np.argmax(q_values)
        return valid_acts[max_idx]

In [33]:
# Hyperparameters
ALPHA = 0.1   # Learning Rate
GAMMA = 0.9   # Discount Factor

def update_q_table(state, action, reward, next_state):
    """Updates the agent's memory using the standard Bellman Optimality Equation."""
    max_future_q = np.max(Q[next_state])
    td_target = reward + GAMMA * max_future_q
    
    # Temporal Difference Update
    Q[state, action] += ALPHA * (td_target - Q[state, action])

In [34]:
# Training parameters
TOTAL_EPISODES = 500
EPSILON = 0.2

print("Training initiated...")

for episode in range(TOTAL_EPISODES):
    current_state = reset(episode)
    done = False
    
    while not done:
        action = choose_action(current_state, EPSILON)
        next_state, reward, done = step(action)
        update_q_table(current_state, action, reward, next_state)
        current_state = next_state

print("Training complete! The Q-Table is fully optimized.")

Training initiated...
Training complete! The Q-Table is fully optimized.


In [35]:
# Reset to standard start configuration
test_state = reset(0) 
done = False
steps_taken = 0

print("Agent is executing the learned optimal policy...")

while not done and steps_taken < 50:
    action = choose_action(test_state, epsilon=0) # Pure exploitation
    next_state, reward, done = step(action)
    
    print(f"Step {steps_taken + 1}: Agent moved to -> ({robu_row}, {robu_col})")
    test_state = next_state
    steps_taken += 1

if (robu_row, robu_col) == GOAL_POSITION:
    print(f"🎉 Success! Goal reached optimally in {steps_taken} steps.")
else:
    print("❌ Failure: Agent failed to locate the goal safely.")
    

Agent is executing the learned optimal policy...
Step 1: Agent moved to -> (0, 1)
Step 2: Agent moved to -> (0, 2)
Step 3: Agent moved to -> (0, 3)
Step 4: Agent moved to -> (0, 4)
Step 5: Agent moved to -> (0, 5)
Step 6: Agent moved to -> (0, 6)
Step 7: Agent moved to -> (0, 7)
🎉 Success! Goal reached optimally in 7 steps.
